# 01 · Vector Validation

**Purpose:** Match candidate building footprints (Overture, GBA, GloBFP) against reference footprints per city and compute detection + geometry metrics (F1, precision/recall, IoU, area bias, count ratio).

**Inputs:** `configs/validation_configs.yaml`, `data/02_interim/aoi_tracker.csv`, and reference + candidate vector files under `data/01_raw/`.

**Outputs:** per-city tile metrics and summaries under `outputs/metrics/`.

**Run order:** after `00_download_data`; before `04_aggregate_global_metrics`.

**Last run:** _(fill in when you run it)_

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/urban_validation/")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/urban_validation")
CONFIG_PATH  = PROJECT_ROOT / "configs/validation_configs.yaml"

# ── Run control ───────────────────────────────────────────────────────────────
# OVERWRITE = True  : re-run every city, even if outputs already exist.
# OVERWRITE = False : skip cities whose sentinel parquet is already present.
#                     Use this to resume after a crash or to add new cities only.
OVERWRITE = False

# Cities to skip unconditionally (e.g. cities that exceed Colab RAM).
# bgd-rohingya has 111K reference buildings from 33 files; it consistently OOMs
# on standard Colab runtimes. Use a high-RAM runtime or process it separately.
CITY_EXCLUDE = [
    "bgd-rohingya",  # ~111K ref buildings, 33 AOI files → OOM on standard Colab
]

# To reprocess ONLY SpaceNet7 cities (new τ=0.25 threshold, item 21a):
# 1. Set OVERWRITE = False (keeps non-SpaceNet7 cities as-is)
# 2. Delete their sentinel files first so the pipeline treats them as incomplete:
#       sentinel = "vector_metrics_tiles_all_datasets.parquet"
#       for city_id in spacenet7_city_ids:
#           p = PROJECT_ROOT / "outputs/metrics" / city_id / sentinel
#           if p.exists(): p.unlink()
# Then re-run this notebook. Only those cities will be reprocessed.

In [ ]:
import sys, os
# Code from GitHub, data from Drive (see colab_bootstrap.py in the repo).
# Drive PROJECT_ROOT/src is a stale hand-copy; never import from it.
import subprocess as _sp
_sp.run(['wget','-q','-O','/content/colab_bootstrap.py','https://raw.githubusercontent.com/GFDRR/urban_validation/fix/pipeline-audit/colab_bootstrap.py'], check=False)
sys.path.insert(0, '/content')
sys.modules.pop('colab_bootstrap', None); assert 'def setup' in open('/content/colab_bootstrap.py').read(), 'bootstrap download failed - check branch/URL'; from colab_bootstrap import setup as _setup
_setup(PROJECT_ROOT)   # clones repo -> sys.path; cwd stays on Drive

In [ ]:
import logging
import yaml
from src.validator import UrbanValidator

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# Patch overwrite flag into config before constructing Validator
with open(CONFIG_PATH) as f:
    _cfg_preview = yaml.safe_load(f)

datasets_preview = _cfg_preview.get("vector", {}).get("datasets", [])
enabled = [d["name"] for d in datasets_preview if d.get("enabled", True)]
print(f"Config: {CONFIG_PATH}")
print(f"Enabled candidate datasets: {enabled}")

In [ ]:
# Instantiate the validator — this reads the config and AOI tracker,
# resolves file paths, and logs how many datasets are queued.
# The overwrite flag from the cell above is injected before loading.
import yaml, copy

with open(CONFIG_PATH) as f:
    _cfg_patched = yaml.safe_load(f)

# Patch root_dir to the actual PROJECT_ROOT on this Colab instance.
# The YAML may contain a stale path if the project was moved on Drive.
_cfg_patched["root_dir"] = str(PROJECT_ROOT)
_cfg_patched.setdefault("output", {})["overwrite"] = OVERWRITE

# Write patched config to a temp file so Validator can read it
import tempfile, os
_tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix=".yaml", delete=False, dir=PROJECT_ROOT / "configs"
)
yaml.dump(_cfg_patched, _tmp)
_tmp.close()
_PATCHED_CONFIG_PATH = _tmp.name

v = UrbanValidator(_PATCHED_CONFIG_PATH)

# Filter out excluded cities
if CITY_EXCLUDE:
    before = len(v.datasets)
    v.datasets = [ds for ds in v.datasets if ds["id"] not in CITY_EXCLUDE]
    print(f"Excluded {before - len(v.datasets)} city(ies): {CITY_EXCLUDE}")

print(f"\nDatasets queued: {len(v.datasets)}")
for ds in v.datasets:
    print(f"  {ds['id']}")

In [ ]:
import pandas as pd

results = v.validate_vector()

# Clean up temp config
try:
    os.unlink(_PATCHED_CONFIG_PATH)
except Exception:
    pass

# Summary
summary = pd.DataFrame(
    [{"aoi": k, "status": "ok" if v else "failed"} for k, v in results.items()]
)
print(f"\nDone — {len(summary)} AOIs processed.\n")
display(summary.groupby("status")["aoi"].count().rename("count").to_frame())
display(summary)